# Well-Centre Analysis: Validating a Semi-Empirical SARF Anchor Rule

This notebook loads a **converged Gaussian-variant** Fock-PARFLM checkpoint
(Phase 5, OpenWebText) and uses its *context-dependent* learned well centres
`mu_k(xi)` to validate closed-form rules for placing the **static** SARF anchors.

**Why:** the Gaussian variant learns `mu_k(xi) = mu_proj(xi)` per context, while
SARF freezes anchors at PMI-peak token embeddings. If a simple analytical rule
(frequency, manifold modes, PCA shell, ...) predicts where the converged Gaussian
centres actually settle, we can choose SARF anchors by that rule instead of the
PMI heuristic.

**Pipeline:** load checkpoint -> extract centre cloud `{mu_k(xi)}` + responsibilities
`w_k(xi)` over held-out OWT -> weighted k-means -> empirical TARGET modes ->
score candidate rules (Chamfer, mass coverage, LM-head decode Jaccard) -> plots.

All heavy lifting lives in `well_centre_analysis.py` (in `parf/`).

In [ ]:
# ── Cell 1: Environment + paths ─────────────────────────────────────
import os, sys, gc, shutil, subprocess, json, time, math
from pathlib import Path

IN_COLAB = 'google.colab' in sys.modules
print(f'IN_COLAB = {IN_COLAB}')

REPO_URL    = 'https://github.com/dimitarpg13/semsimula-paper.git'
REPO_BRANCH = 'main'


def _sh(cmd):
    print(f'$ {cmd}')
    r = subprocess.run(cmd, shell=True)
    if r.returncode != 0:
        raise RuntimeError(f'exit {r.returncode}: {cmd}')


if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive', force_remount=False)
    REPO_ROOT = Path('/content/semsimula-paper')
    if not (REPO_ROOT / '.git').exists():
        if REPO_ROOT.exists():
            shutil.rmtree(REPO_ROOT)
        _sh(f'git clone --depth 1 --branch {REPO_BRANCH} {REPO_URL} {REPO_ROOT}')
    GDRIVE_ROOT = Path('/content/drive/MyDrive/semsimula_fock_gaussian_sarf_openwebtext_phase5')
    DATA_DIR = GDRIVE_ROOT / 'data'
    CKPT_DIR = GDRIVE_ROOT / 'checkpoints'
    _sh('pip install -q transformers')
else:
    REPO_ROOT = Path('.').resolve()
    while not (REPO_ROOT / '.git').exists() and REPO_ROOT != REPO_ROOT.parent:
        REPO_ROOT = REPO_ROOT.parent
    DATA_DIR = REPO_ROOT / 'notebooks' / 'conservative_arch' / 'data'
    CKPT_DIR = REPO_ROOT / 'notebooks' / 'conservative_arch' / 'scaleup' / 'results' / 'phase5' / 'ckpts'

CA_DIR = REPO_ROOT / 'notebooks' / 'conservative_arch'
for sub in ['', 'parf', 'multixi', 'scaleup', 'sarf_mass_variant', 'energetic_minima']:
    d = str(CA_DIR / sub) if sub else str(CA_DIR)
    if d not in sys.path:
        sys.path.insert(0, d)

# Override CKPT_PATH directly if your checkpoint lives elsewhere.
CKPT_PATH = CKPT_DIR / 'fock_gaussian_sarf_owt_phase5_best.pt'
print(f'DATA_DIR  = {DATA_DIR}')
print(f'CKPT_PATH = {CKPT_PATH}')
print(f'  exists = {CKPT_PATH.exists()}')

In [ ]:
# ── Cell 2: Load checkpoint + rebuild Gaussian model ────────────────────
import torch
import numpy as np
from model_fock_parf_multixi import FockMultiXiPARFLM, FockMultiXiPARFConfig
from model_gaussian_vtheta import MixtureGaussianVTheta, GaussianVThetaMultiXiAdapter

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'DEVICE = {DEVICE}')

ckpt = torch.load(str(CKPT_PATH), map_location='cpu')
variant = ckpt.get('train_cfg', {}).get('v_theta_variant', 'gaussian')
print(f"checkpoint: step={ckpt.get('step')}  val_ppl={ckpt.get('val_ppl')}  variant={variant}")
assert variant == 'gaussian', (
    f"This analysis requires the *Gaussian* variant (learned centres); got {variant!r}. "
    'SARF has static anchors -- there is nothing to extract.')

cfg = FockMultiXiPARFConfig(**ckpt['model_cfg'])
model = FockMultiXiPARFLM(cfg).to(DEVICE)

# Infer K_mix from the saved mu_proj shape: (K*d, xi_d).
sd = ckpt['model_state_dict']
mu_w = sd['V_theta.inner.mu_proj.weight']
K_MIX = mu_w.shape[0] // cfg.d
XI_CHANNELS = cfg.xi_channels
xi_d = XI_CHANNELS * cfg.d
print(f'  d={cfg.d}  L={cfg.L}  K_xi={XI_CHANNELS}  K_mix={K_MIX}')

inner = MixtureGaussianVTheta(
    d=cfg.d, K=K_MIX, w_scale=1.0, xi_d=xi_d,
    init_log_precision=-math.log(cfg.d), precision_max=2.0 / cfg.d,
)
model.V_theta = GaussianVThetaMultiXiAdapter(inner, K=XI_CHANNELS, d=cfg.d).to(DEVICE)

missing, unexpected = model.load_state_dict(sd, strict=True)
model.eval()
print('  state_dict loaded OK')
E = model.E.weight.data.detach().cpu()
VOCAB_SIZE = cfg.vocab_size

In [ ]:
# ── Cell 3: Held-out OpenWebText (val) tokens ────────────────────────
from transformers import AutoTokenizer
tok = AutoTokenizer.from_pretrained('gpt2')

MAX_TRAIN_TOKENS = 200_000_000
VAL_TOKENS       = 2_000_000
val_cache = DATA_DIR / f'openwebtext_val_{VAL_TOKENS // 1_000_000}M.npy'

if not val_cache.exists():
    # Search prior-phase caches (same logic as the Phase 5 notebook).
    for alt_name in [
        'semsimula_fock_structured_vtheta_owt_phase4',
        'semsimula_fock_gaussian_sarf_openwebtext_phase5',
        'semsimula_splm_openwebtext_phase4',
    ]:
        alt = (Path(f'/content/drive/MyDrive/{alt_name}/data') if IN_COLAB
               else Path.home() / alt_name / 'data')
        cand = alt / f'openwebtext_val_{VAL_TOKENS // 1_000_000}M.npy'
        if cand.exists():
            val_cache.parent.mkdir(parents=True, exist_ok=True)
            shutil.copy2(str(cand), str(val_cache))
            print(f'Reused val cache from {alt}')
            break

assert val_cache.exists(), (
    f'val cache not found at {val_cache}. Run the Phase 5 data cell first to '
    'materialise the OpenWebText cache, or point DATA_DIR at it.')
val_ids = np.load(str(val_cache))
print(f'val: {len(val_ids):,} tokens')

In [ ]:
# ── Cell 4: Extract centre cloud + empirical TARGET modes ───────────────
import well_centre_analysis as wca
import importlib; importlib.reload(wca)

N_BATCHES  = 64        # tokens sampled = N_BATCHES * BATCH * BLOCK
BATCH      = 8
BLOCK      = 512
MAX_POINTS = 200_000   # subsample of (token x component) centres kept
N_S        = 64        # number of anchors / target modes (match SARF_N_ANCHORS)

cloud = wca.extract_centre_cloud(
    model, val_ids, n_batches=N_BATCHES, batch_size=BATCH, block_size=BLOCK,
    device=DEVICE, max_points=MAX_POINTS, seed=0,
)
print(f"cloud: mu={tuple(cloud['mu'].shape)}  w={tuple(cloud['w'].shape)}  "
      f"h_L={tuple(cloud['h_L'].shape)}")

# Active-component diagnostic: mean responsibility per component.
mean_resp = cloud['w'].mean(0)
active = (mean_resp > (1.0 / (4 * K_MIX))).sum().item()
print(f'mean responsibility per component: '
      f"{[round(float(x),3) for x in mean_resp]}")
print(f'  K_eff (resp > 1/(4K)) = {active} / {K_MIX}')

points, weights = wca.flatten_weighted_cloud(cloud)
target_modes = wca.weighted_kmeans(points, weights, N_S, seed=0)
print(f'TARGET modes: {tuple(target_modes.shape)}')

In [ ]:
# ── Cell 5: Build candidate anchors + score against TARGET ──────────────
# Surprisal from the val unigram (matches the logfreq mass convention).
counts = wca.compute_token_counts(val_ids, VOCAB_SIZE)
_p = (counts + 1.0) / (counts.sum() + VOCAB_SIZE)
surprisal = -np.log(_p)

RULES = ['pmi', 'freq', 'hmodes', 'pca', 'surprisal']
anchors = {}
for rule in RULES:
    anchors[rule] = wca.build_rule_anchors(
        rule, E=E, n_anchors=N_S, ids=val_ids, vocab_size=VOCAB_SIZE,
        cloud_points=points, cloud_weights=weights, h_L=cloud['h_L'],
        surprisal=surprisal, seed=0,
    )
    print(f'  built {rule:<10} anchors {tuple(anchors[rule].shape)}')

# Shared coverage sigma from the TARGET (identical threshold for all rules).
cloud_n = wca.standardize_rows(points)
modes_n = wca.standardize_rows(target_modes)
SIGMA = wca.default_sigma(cloud_n, modes_n)
print(f'shared sigma = {SIGMA:.4f}')

scores = {}
for rule in RULES:
    scores[rule] = wca.score_rule(
        anchors[rule], target_modes, points, weights, E,
        sigma=SIGMA, decode_top_k=1,
    )

print(f"\n{'rule':<11}{'chamfer\u2193':>12}{'coverage\u2191':>12}{'jaccard\u2191':>12}")
for rule in RULES:
    s = scores[rule]
    print(f"{rule:<11}{s['chamfer']:>12.4f}{s['coverage']:>12.3f}{s['decode_jaccard']:>12.3f}")

best_chamfer = min(RULES, key=lambda r: scores[r]['chamfer'])
best_cover   = max(RULES, key=lambda r: scores[r]['coverage'])
print(f'\nbest chamfer:  {best_chamfer}')
print(f'best coverage: {best_cover}')

In [ ]:
# ── Cell 6: Plots ─ PCA scatter (cloud vs anchors) + metric bars ────────
import matplotlib.pyplot as plt

# 2-D PCA fit on the standardised cloud (subsample for speed).
_sub = points[torch.randperm(points.shape[0])[:20000]]
_subn = wca.standardize_rows(_sub)
_mean = _subn.mean(0, keepdim=True)
_U, _S, _V = torch.pca_lowrank(_subn - _mean, q=2)


def project(x):
    return ((wca.standardize_rows(x) - _mean) @ _V[:, :2]).numpy()


cloud_xy = project(_sub)
modes_xy = project(target_modes)

fig, axes = plt.subplots(2, 3, figsize=(16, 9))
for ax, rule in zip(axes.flat, RULES):
    ax.scatter(cloud_xy[:, 0], cloud_xy[:, 1], s=2, alpha=0.15, label='centre cloud')
    a_xy = project(anchors[rule])
    ax.scatter(modes_xy[:, 0], modes_xy[:, 1], s=40, marker='x', label='TARGET modes')
    ax.scatter(a_xy[:, 0], a_xy[:, 1], s=40, marker='^', label=f'{rule} anchors')
    ax.set_title(f"{rule}  (chamfer={scores[rule]['chamfer']:.2f}, "
                 f"cov={scores[rule]['coverage']:.2f})")
    ax.legend(fontsize=7, loc='upper right')

# Metric bar chart in the last panel.
ax = axes.flat[5]
xs = np.arange(len(RULES))
ax.bar(xs - 0.2, [scores[r]['coverage'] for r in RULES], width=0.4, label='coverage')
ax.bar(xs + 0.2, [scores[r]['decode_jaccard'] for r in RULES], width=0.4, label='jaccard')
ax.set_xticks(xs); ax.set_xticklabels(RULES, rotation=30)
ax.set_title('coverage / decode-jaccard (higher better)'); ax.legend(fontsize=8)
plt.tight_layout()
_out = CA_DIR / 'scaleup' / 'results' / 'well_centre_rule_scores.png'
_out.parent.mkdir(parents=True, exist_ok=True)
plt.savefig(str(_out), dpi=120, bbox_inches='tight')
print(f'saved {_out}')
plt.show()

In [ ]:
# ── Cell 7: Decode comparison ─ what tokens do centres vs rules point to? ──
def decode_tokens(vecs, top_k=1, n_show=24):
    ids_dec, _ = wca.decode_via_lm_head(wca.standardize_rows(vecs), wca.standardize_rows(E), top_k=top_k)
    toks = [tok.decode([int(i)]).strip() for i in ids_dec[:, 0].tolist()]
    # de-dup preserving order, for a compact readout
    seen, out = set(), []
    for t in toks:
        if t not in seen:
            seen.add(t); out.append(t)
        if len(out) >= n_show:
            break
    return out

print('TARGET modes decode to:')
print('  ', decode_tokens(target_modes))

# Per-component converged centroid (mean over contexts of mu_k), decoded.
per_comp_centroid = cloud['mu'].mean(0)  # (K, d)
print('\nPer-component centroids (mean_xi mu_k) decode to:')
comp_ids, _ = wca.decode_via_lm_head(wca.standardize_rows(per_comp_centroid), wca.standardize_rows(E), top_k=3)
for k in range(K_MIX):
    toks = [tok.decode([int(i)]).strip() for i in comp_ids[k].tolist()]
    print(f'  basin {k}: resp={float(mean_resp[k]):.3f}  -> {toks}')

print('\nEach rule\'s anchors decode to:')
for rule in RULES:
    print(f'  {rule:<10}: {decode_tokens(anchors[rule])}')

## Interpreting the results

- **Chamfer (lower)**: how close the rule's anchors sit to the empirical TARGET
  modes in the shared standardised basis.
- **Coverage (higher)**: responsibility-weighted fraction of the centre cloud
  within `sigma` of some anchor -- does the rule's anchor set actually blanket
  where the model puts its mass?
- **Decode Jaccard (higher)**: do the rule's anchors decode (via the LM head) to
  the same tokens as the empirical modes?

**Decision:** the rule that minimises Chamfer while maximising coverage is the
candidate semi-empirical law. Record the table + decode comparison in
`companion_notes/SARF_Anchor_Placement_From_Converged_Gaussian_Centres.md`.

**Definitive follow-up (out of scope here):** train short SARF runs using the
top one or two rules' anchors and compare best val PPL against PMI-peak SARF and
the learned Gaussian. If a rule wins, add it as a new `anchor_mode` in the
Phase 5 SARF builder.